In [ ]:
#Task 4.1 — One validated call
!pip install -q google-genai pydantic pandas

import json
import time
import getpass
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

# 1. Setup Client
API_KEY = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

# Use the exact Flash model from your Session 22.1 notebook (e.g., "gemini-2.5-flash")
MODEL_NAME = "gemini-3.5-flash"

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

# Track total API calls across Tasks 4.1 - 4.3
API_CALL_COUNT = 0

# 2. JSON Schema (root_cause is a plain string, NOT an enum)
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

# 3. System Instruction & CRAFT Prompt Builder
SYSTEM_TEXT = (
    "You are a healthcare operations analyst. You read short call-centre follow-up notes "
    "about missed clinic appointments—including English and Hinglish—and extract structured "
    "insights strictly following the user's rules and JSON schema."
)

def build_craft_prompt(note_text: str) -> str:
    return f"""
Context: A clinic call centre contacts patients who missed their scheduled appointments and logs brief notes (sometimes in Hinglish or informal English).
Role: Act as a clinic operations analyst classifying the primary reason for the missed appointment.
Action: Read the note below and return a JSON object with:
  - root_cause: MUST be strictly one of these exact strings: {", ".join(ALLOWED_ROOT_CAUSES)}.
    Rule: If the note is vague, ambiguous, or does not clearly match a specific reason, choose "other" rather than guessing.
  - confidence: a float between 0.0 and 1.0 reflecting how certain you are.
  - summary: a concise summary of the note in at most 25 words.
  - recommended_action: one practical next step for the clinic staff.
Tone: Objective, concise, and operational.
Format: Valid JSON matching the schema.

Call-centre note: "{note_text}"
""".strip()

# 4. Core Analysis Function
def analyse_note(note_text: str, return_raw: bool = False):
    global API_CALL_COUNT
    prompt = build_craft_prompt(note_text)

    config_kwargs = dict(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(**config_kwargs),
        )
    except Exception as e:
        # Fallback if the specific model rejects thinking_config
        if "thinking" in str(e).lower():
            config_kwargs.pop("thinking_config", None)
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(**config_kwargs),
            )
        else:
            raise

    API_CALL_COUNT += 1
    raw_text = response.text
    data = json.loads(raw_text)

    if return_raw:
        return data, raw_text
    return data

# 5. Run on note N001 and print the parsed dictionary
notes_df = pd.read_csv("followup_notes.csv")
# Adjust 'note_text' below if your column has a different name (e.g., 'note' or 'notes')
NOTE_COL = "note_text"

n001_text = notes_df.loc[notes_df["note_id"] == "N001", NOTE_COL].values[0]
n001_result = analyse_note(n001_text)
print("N001 Parsed Dictionary:")
print(json.dumps(n001_result, indent=2))

Paste your Gemini API key: ··········
N001 Parsed Dictionary:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.95,
  "summary": "Patient forgot appointment due to not receiving a reminder message and requested to rebook for next week.",
  "recommended_action": "Reschedule appointment for next week and verify patient contact details for reminder notifications."
}


In [ ]:
#Task 4.2 — Validate and process the 10 notes

In [ ]:
# (a) Pydantic Model & Allowed-List Check
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def is_allowed_root_cause(value: str) -> bool:
    return value in ALLOWED_ROOT_CAUSES

# (b), (c), (d) Loop over first 10 rows, validate, sleep, and save
first_10 = notes_df.head(10).copy()
results = []

for idx, row in first_10.iterrows():
    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row[NOTE_COL]

    # Input validation: check NaN first, then empty/whitespace string
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": "",
        })
        print(f"{note_id}: skipped_blank (no API call)")
        continue

    raw_text = ""
    data = {}
    status = "invalid"

    try:
        data, raw_text = analyse_note(str(note_text), return_raw=True)
        # Validate with Pydantic and allowed-list check
        validated = FollowupInsight(**data)
        if is_allowed_root_cause(validated.root_cause):
            status = "valid"
        else:
            status = "invalid"
    except Exception as e:
        print(f"{note_id}: validation/API error -> {e}")
        status = "invalid"

    # Keep every reply as-is; never repair missing fields
    results.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": data.get("root_cause", "") if isinstance(data, dict) else "",
        "confidence": data.get("confidence", "") if isinstance(data, dict) else "",
        "summary": data.get("summary", "") if isinstance(data, dict) else "",
        "recommended_action": data.get("recommended_action", "") if isinstance(data, dict) else "",
        "validation_status": status,
        "raw_response": raw_text,
    })
    print(f"{note_id}: {status} -> {results[-1]['root_cause']}")

    # Respect free-tier rate limit between API calls
    time.sleep(13)

analysed_df = pd.DataFrame(results, columns=[
    "note_id", "appointment_id", "root_cause", "confidence",
    "summary", "recommended_action", "validation_status", "raw_response"
])

analysed_df.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv with shape:", analysed_df.shape)
display(analysed_df[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]])

N001: valid -> forgot_or_no_reminder
N002: valid -> distance_or_transport
N003: valid -> cost_or_fee
N004: skipped_blank (no API call)
N005: valid -> illness_or_emergency
N006: valid -> work_or_schedule
N007: valid -> forgot_or_no_reminder
N008: validation/API error -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
N008: invalid -> 
N009: valid -> other
N010: valid -> work_or_schedule

Saved notes_analysed.csv with shape: (10, 8)


,note_id,appointment_id,root_cause,confidence,validation_status
0,N001,A0872,forgot_or_no_reminder,0.95,valid
1,N002,A0897,distance_or_transport,0.95,valid
2,N003,A0526,cost_or_fee,0.98,valid
3,N004,A0926,other,0.0,skipped_blank
4,N005,A0534,illness_or_emergency,0.98,valid
5,N006,A0507,work_or_schedule,0.95,valid
6,N007,A0333,forgot_or_no_reminder,0.95,valid
7,N008,A0222,,,invalid
8,N009,A0977,other,0.95,valid
9,N010,A0596,work_or_schedule,0.95,valid


In [ ]:
#Task 4.3 — Break it on purpose

In [ ]:
# (a) Feed Pydantic two deliberately bad replies
print("=== Task 4.3(a): Deliberate Pydantic Failures ===")

bad_confidence = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment date.",
    "recommended_action": "Send SMS reminder 24h prior."
}

missing_action = {
    "root_cause": "cost_or_fee",
    "confidence": 0.9,
    "summary": "Patient could not afford consultation fee."
    # recommended_action intentionally omitted
}

for label, payload in [("Confidence = 1.7", bad_confidence), ("Missing recommended_action", missing_action)]:
    print(f"\n--- Testing: {label} ---")
    try:
        FollowupInsight(**payload)
    except ValidationError as e:
        print(e)

# (b) Compare V1 vs V2 prompts on N007 (Hinglish) and N009 (ambiguous)
print("\n=== Task 4.3(b): Prompt Comparison (V1 vs V2) ===")

def run_custom_prompt(prompt_text: str):
    global API_CALL_COUNT
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),
        ),
    )
    API_CALL_COUNT += 1
    return json.loads(response.text)

comparison_rows = []
for target_id in ["N007", "N009"]:
    text = str(notes_df.loc[notes_df["note_id"] == target_id, NOTE_COL].values[0])

    # V1: Minimal prompt
    v1_prompt = f'Classify this note: "{text}"'
    v1_out = run_custom_prompt(v1_prompt)
    time.sleep(13)

    # V2: Full CRAFT prompt
    v2_prompt = build_craft_prompt(text)
    v2_out = run_custom_prompt(v2_prompt)
    time.sleep(13)

    for version, out in [("V1 (Basic)", v1_out), ("V2 (CRAFT)", v2_out)]:
        rc = out.get("root_cause", "")
        comparison_rows.append({
            "note_id": target_id,
            "prompt_version": version,
            "root_cause": rc,
            "in_allowed_list": is_allowed_root_cause(rc),
            "confidence": out.get("confidence", ""),
            "summary": out.get("summary", ""),
        })

comp_df = pd.DataFrame(comparison_rows)
display(comp_df)

print("\nDeployment Choice: Deploy V2 (CRAFT). V1 invents free-text labels outside the allowed list "
      "or guesses on ambiguous notes, whereas V2 constrains root_cause to the allowed taxonomy "
      "and safely defaults to 'other' when uncertain.")

# (c) Total API call count
print(f"\n=== Task 4.3(c): Total API Calls Used (Tasks 4.1–4.3) ===\nTotal calls: {API_CALL_COUNT}")

=== Task 4.3(a): Deliberate Pydantic Failures ===

--- Testing: Confidence = 1.7 ---
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

--- Testing: Missing recommended_action ---
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ford consultation fee.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

=== Task 4.3(b): Prompt Comparison (V1 vs V2) ===


,note_id,prompt_version,root_cause,in_allowed_list,confidence,summary
0,N007,V1 (Basic),System Error / Reminders,False,0.95,Patient forgot the appointment because no SMS ...
1,N007,V2 (CRAFT),forgot_or_no_reminder,True,0.95,Patient forgot the appointment because no SMS ...
2,N009,V1 (Basic),Unreachable Patient,False,0.95,Patient phone was switched off; unreachable af...
3,N009,V2 (CRAFT),other,True,0.95,Patient could not be reached after three attem...



Deployment Choice: Deploy V2 (CRAFT). V1 invents free-text labels outside the allowed list or guesses on ambiguous notes, whereas V2 constrains root_cause to the allowed taxonomy and safely defaults to 'other' when uncertain.

=== Task 4.3(c): Total API Calls Used (Tasks 4.1–4.3) ===
Total calls: 14


In [ ]:
#Task 4.4 — Recommendation and GitHub

In [ ]:
appointments_df = pd.read_csv("appointments_clean.csv")
analysed_df = pd.read_csv("notes_analysed.csv")

merged_df = pd.merge(analysed_df, appointments_df, on="appointment_id", how="inner")
print("Merged table shape:", merged_df.shape)  # Should be (10, ...)

ct = pd.crosstab(merged_df["root_cause"], merged_df["reminder_sent"], margins=True)
print("\nCross-tabulation: root_cause vs reminder_sent")
display(ct)

Merged table shape: (10, 23)

Cross-tabulation: root_cause vs reminder_sent


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,0,1
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,4,9
